# Kalman Filterを用いた線形システムのパラメータ推定

Kalman Filterを用いて、線形システムのパラメータの推定を行う。



## 離散モデル

次の線形モデルの$m[k], D[k]$を測定値の$f[k], a[k], v[k]$から求めることを考える。

$$
f[k] = m[k] \ a[k] + D[k] \  v[k] + w[k]
$$

上式は、慣性 $m[n]$と、粘性 $D[n]$ による運動モデルである。そして、$w[n]$は測定値に含まれるノイズを表した観測雑音であり、平均$0$、分散$\sigma_w^2$の正規分布に従う。

## 推定モデル

推定すべきパラメータを$\hat{\theta}$、 システムの応答値を$z[k]$、システムへの入力を$y[k]$とする。

$$
\hat{\theta}[k] = \begin{bmatrix} m[k] \\ D[k] \end{bmatrix}, \quad z[k] = \begin{bmatrix} a[k] \\ v[k] \end{bmatrix} , \quad y[k] = f[k]
$$

すると、離散モデルは次の形式で記述できる。

$$
y[k] = \hat{\theta}^T z[k] + w[k]
$$

$\hat{\theta}[k]$は時変であるため、次のランダムウォークするウィナーモデルで表現することができる。<br>
$r[k]$はスカラーであり、平均$0$、分散$\sigma_r^2$の正規分布に従う。

$$
\hat{\theta}[k+1] = \hat{\theta}[k] + b r[k]
$$

$r[k]$はスカラーであるため、$\hat{\theta}$の各要素に均一に影響すると考えると、$b=[1,1]^T$となる。

$$
\begin{bmatrix}
m[k+1] \\ D[k+1] 
\end{bmatrix} =
\begin{bmatrix}
m[k] \\ D[k] 
\end{bmatrix} +
\begin{bmatrix}
1 \\ 1 
\end{bmatrix} r[k]
$$

## Kalman Fitlerによる推定アルゴリズム

### 初期値を次のように設定する

- $\hat{\theta}[0] = \theta_0 \in \mathrm{R}^{2 \times 1}$
    - $\theta_0$は事前にある程度の値が分かればそれを設定する。
    - 事前情報が利用できない場合は $\theta_0 = 0$ とする。
- $P[0] = \alpha \mathrm{I}$ (2 x 2 の対角行列)
    - $\alpha >0$ であり、通常は$10^3$程度の値とする。

### アルゴリズム

各時刻で次の式を上から順番に計算していくことで、推定パラメータ$\hat{\theta}[k]$を求める。

$$
\begin{aligned}
P^{-}[k] &= P[k-1] + \sigma_r^2 \ b b^T \\
\epsilon[k] &= y[k] - z[k]^T \theta[k-1] \\
g[k] &= \frac{P^{-}[k] \ z[k]}{z[k]^T P^{-} \ z[k] + \sigma_w^2} \\
\hat{\theta}[k] &= \hat{\theta}[k-1] + g[k] \epsilon[k] \\
P[k] &= (\mathrm{I} - g[k] z[k]^T)P^{-}[k]
\end{aligned}
$$

- $P[k]$ : 真値と推定値の誤差の共分散行列
- $P^{-}[k]$ 時刻$k$における誤差共分散行列の推定値 (2 x 2 の行列)
- $\epsilon[k]$ : 陳地と推定値の誤差
- $g[k]$ : カルマンゲイン ( 2 x 1 の縦ベクトル)